<a href="https://colab.research.google.com/github/ksusmitha879-cyber/FlyrankStarter/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ksusmitha879-cyber/FlyrankStarter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

I picked this lane because a first look at the starter dataset already shows a strong, non-obvious pattern
here: CTR does not fade smoothly as position gets worse — it falls off a cliff after the top 3 spots, then
stays low and roughly flat for everything else. That is exactly the shape Lane 4 is built around: "compare
pages only to others in the same position tier," not across all positions at once. It also gives me a
concrete decision-support output — a ranked list of visible, well-positioned pages that are under-capturing
clicks relative to their own tier's peers — which is something a content reviewer could actually act on with
limited time. I am keeping Lane 2 (Refresh Scoring) as a mental backup, since the two lanes share a lot of
the same features and baseline logic, but Lane 4's question is sharper and better supported by what I've
already seen in the data (Section 3).

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
**The question:** Which visible, decently-positioned pages are under-capturing clicks relative to other
pages in the same position tier — and therefore deserve a title / meta description / snippet review?

**Decision it improves:** which pages an SEO/content reviewer should look at *first* out of hundreds of
visible pages, given they only have time to review a handful per week.

**Who acts, and what they do:** a content strategist or SEO reviewer with limited review capacity. Given
my ranked list with reason codes, they open the top-ranked pages and check the title tag, meta description,
and search snippet for a possible rewrite — this is metadata review, not a guarantee of a rewrite.

**Cost of a wrong call:**
- *False positive* (I rank a page high, but it wasn't really underperforming): wastes a reviewer's limited
  time on a page that was already fine, while a real opportunity elsewhere goes unreviewed one more week.
  Costly in opportunity terms, not catastrophic.
- *False negative* (a real under-capturing page never surfaces): the page keeps quietly losing clicks it
  could have earned, and nobody ever looks at it because it never appears on the list.
- Because capacity is the scarce resource, precision near the top of the ranked list matters more than
  catching every possible case — a few wrong picks in the top 20 waste real reviewer hours.

**Why data or ML helps at all:** a flat rule like "flag anything with CTR under 0.5%" is not enough, because
"low CTR" only means something *relative to position* — a 0.3% CTR is normal at page 3-5 but a real gap at
top-3. Comparing every page to *its own tier's* typical CTR (Section 3) already surfaces a real, sizeable
gap. Whether a learned model can rank that gap better than a same-tier-median rule is exactly the honest
baseline-vs-model comparison I'll run in later weeks — I am not assuming ML wins before testing it.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv('/content/content_refresh_anonymized.csv')

# avg_position == 0 means "no position data", not rank zero -- exclude those rows (per data dictionary)
valid = df[df['avg_position'] > 0].copy()
print(f"Rows with real position data: {len(valid):,} of {len(df):,} total rows")
print()

# CTR by position tier -- this is the core pattern behind my lane choice
tier_order = ['top_3', 'striking', 'page_1', 'page_3_5', 'deep']
ctr_by_tier = valid.groupby('position_tier')['ctr'].agg(['count', 'mean', 'median']).reindex(tier_order).round(3)
print("Mean/median CTR (%) by position tier:")
print(ctr_by_tier)
print()

# How many visible pages already sit below their OWN tier's median CTR?
min_impressions = 500
tier_median_ctr = valid.groupby('position_tier')['ctr'].median()
valid['tier_median_ctr'] = valid['position_tier'].map(tier_median_ctr)
visible = valid[valid['impressions_90d'] >= min_impressions]
under_median = visible[visible['ctr'] < visible['tier_median_ctr']]
print(f"Visible pages (impressions_90d >= {min_impressions}): {len(visible):,}")
print(f"Of those, below their OWN tier's median CTR: {len(under_median):,} "
      f"({100 * len(under_median) / len(visible):.1f}%)")

Rows with real position data: 28,795 of 30,000 total rows

Mean/median CTR (%) by position tier:
               count   mean  median
position_tier                      
top_3           1116  2.764    0.00
striking        7304  0.323    0.11
page_1         11814  0.652    0.16
page_3_5        7242  0.222    0.03
deep            1319  0.150    0.00

Visible pages (impressions_90d >= 500): 16,726
Of those, below their OWN tier's median CTR: 4,973 (29.7%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

**What I can claim, based on Section 3's numbers:**
- Observed: in this 30,000-row starter slice, average CTR is strongly and non-linearly associated with
  position tier — mean CTR at top_3 (2.76%) is roughly 4x higher than page_1 (0.65%) and 8-9x higher than
  page_3_5 or deeper. This is a real, measured pattern in this data, not an assumption.
- Observed: about a third of visible pages (impressions_90d >= 500) already sit below their own tier's
  median CTR — a same-tier comparison gap, which is the raw material my ranked opportunity list will be
  built from.
- Directional / decision-support: a ranked list built from this gap can tell a reviewer *where to look
  first*, given limited time. That is a prioritization tool, not a certainty.

**What I cannot and will not claim:**
- I cannot claim that rewriting a title or meta description *will* raise CTR — that would be a causal claim,
  and this data is observational. Only an experiment (e.g. a before/after test on changed pages) could show
  that.
- I cannot claim to predict or reverse-engineer Google's ranking or CTR algorithm. `avg_position` and `ctr`
  are outcomes I observe, not mechanisms I understand.
- I cannot yet claim the "under median CTR" gap is really about title/meta quality specifically — it could
  also reflect intent mismatch, consolidation with a sibling page, seasonality, or plain noise on low-volume
  pages. Ruling those out is exactly what the signal audit and leakage checks in later weeks are for.
- I am excluding the 1,205 rows where `avg_position == 0` from all of this, since that means "no position
  data," not "rank zero" — mixing them in would silently distort every tier average.


## Self-check

Before you submit, confirm each line honestly:

- [ done] Every section above is filled — markdown thinking AND the code that backs it
- [done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ done] No client names, URLs, or private queries anywhere
- [ done] My claims use careful words: observed, measured, directional, decision-support
- [done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.